# Практика 1. Python с нуля до запроса к LLM

**Курс «Разработка LLM-приложений»**

План занятия:
1. **Основы Python** — типы, строки, списки, словари, ветвления, циклы, функции.
2. **Классический NLP** — морфология (pymorphy3), извлечение сущностей (Natasha), классификатор на TF-IDF (scikit-learn).
3. **Запрос к LLM через `requests`** — та же задача, решённая языковой моделью.

Весь семинар мы работаем с одними данными: **отзывы покупателей** и **текст договора** из лекции. Одну и ту же задачу решаем тремя способами — правилами, классическим ML и LLM.

**Как работать с ноутбуком**
- Ячейка запускается `Shift + Enter`.
- Ячейки с ✍️ — упражнения. Под каждой есть свёрнутая ячейка **«Решение»**: сначала попробуйте сами.
- Запустите ячейку подготовки ниже сразу: библиотеки скачиваются около минуты.

## 0. Подготовка

In [ ]:
!pip install -q pymorphy3 natasha
print("Готово")

In [ ]:
# Данные для всего занятия

reviews = [
    ("Отличный чайник, закипает быстро, очень доволен", "pos"),
    ("Доставили быстро, упаковка целая, всё работает", "pos"),
    ("Хороший товар за свои деньги, рекомендую", "pos"),
    ("Удобная ручка, тихо работает, качество отличное", "pos"),
    ("Пользуюсь месяц, нареканий нет, покупкой доволен", "pos"),
    ("Красивый дизайн, кухня сразу преобразилась", "pos"),
    ("Курьер вежливый, привёз раньше срока, спасибо", "pos"),
    ("Отличное качество сборки, ничего не люфтит", "pos"),
    ("Супер, взяли второй такой же родителям", "pos"),
    ("Работает отлично, подсветка красивая, советую", "pos"),
    ("Хорошая модель, вода не пахнет пластиком", "pos"),
    ("Быстро греет, удобно наливать, всем рекомендую", "pos"),
    ("Цена приятная, качество хорошее, доволен покупкой", "pos"),
    ("Отличный подарок, мама в восторге", "pos"),
    ("Всё супер, магазин надёжный", "pos"),
    ("Сломался через неделю, ужасное качество", "neg"),
    ("Привезли с трещиной на корпусе, брак", "neg"),
    ("Пахнет пластиком, пить невозможно", "neg"),
    ("Доставка задержалась на две недели, кошмар", "neg"),
    ("Плохая сборка, крышка болтается", "neg"),
    ("Протекает снизу, вернул продавцу", "neg"),
    ("Очень шумный и долго греет, не советую", "neg"),
    ("Кнопка перестала работать на третий день", "neg"),
    ("Ужасный сервис, деньги не вернули", "neg"),
    ("Товар не соответствует описанию, разочарован", "neg"),
    ("Плохое качество пластика, появились пятна", "neg"),
    ("Пришёл не тот цвет, менять отказались", "neg"),
    ("Отвратительная упаковка, всё помято", "neg"),
    ("Не включается, брак, деньги на ветер", "neg"),
    ("Сгорел при первом включении, опасно", "neg"),
]

# Отзывы для проверки — формулировок из них нет в обучающих данных
test_reviews = [
    ("Отличная доставка, чайник хороший", "pos"),
    ("Привезли быстро, но через неделю он сломался", "neg"),
    ("Вроде нормальный, но ручка греется так, что не взять", "neg"),
    ("Не ожидал, что за такие деньги будет так хорошо", "pos"),
    ("Нареканий нет — разве что к курьеру, который опоздал", "pos"),
    ("Ни разу не пожалел о покупке", "pos"),
]

contract_text = (
    "Договор поставки № 17/26 от 03.02.2026. "
    "ООО «Северная мельница» (Поставщик) в лице генерального директора Иванова Петра Сергеевича "
    "обязуется поставить ООО «Вкусный дом» (Покупатель) муку пшеничную высшего сорта "
    "на сумму 1 240 000 руб. в течение 10 рабочих дней. Место поставки: г. Кострома."
)

print(f"Отзывов: {len(reviews)}, тестовых: {len(test_reviews)}")

# Часть 1. Основы Python

## 1.1. Первые шаги и базовые типы

В Python **отступы — часть синтаксиса**: блок кода определяется отступом, а не фигурными скобками.
Тип переменной не объявляется — он определяется значением (*динамическая типизация*).

In [ ]:
# Комментарий начинается с #
print("Привет, Python!")

age = 20            # int — целое число
price = 1499.90     # float — дробное
name = "Анна"       # str — строка
is_student = True   # bool — True / False
nothing = None      # None — «нет значения»

print(type(age), type(price), type(name), type(is_student), type(nothing))

In [ ]:
# Арифметика
print(7 / 2)    # 3.5 — обычное деление всегда даёт float
print(7 // 2)   # 3   — целочисленное деление
print(7 % 2)    # 1   — остаток
print(2 ** 10)  # 1024 — степень

# Преобразование типов
print(int("42") + 1)
print(str(42) + "1")   # склейка строк, а не сложение

### Строки

In [ ]:
review = "  Отличный чайник, закипает быстро!  "

print(review.strip())                 # убрать пробелы по краям
print(review.lower())                 # в нижний регистр
print(review.replace("чайник", "товар"))
print(review.strip().split())         # разбить на слова -> список

words = ["мука", "пшеничная", "высший сорт"]
print(", ".join(words))               # склеить список в строку

# f-строки: подстановка значений
print(f"{name}, вам {age} лет, товар стоит {price} руб.")

# Индексы и срезы
s = "Python"
print(s[0], s[-1])    # первый и последний символ
print(s[0:3])         # 'Pyt' — с 0 по 2 включительно
print(s[::-1])        # строка задом наперёд

✍️ **Упражнение 1.1.** Возьмите первый отзыв `reviews[0][0]`. Выведите:
1. отзыв в нижнем регистре;
2. количество слов в нём;
3. первое слово.

In [ ]:
# ✍️ Ваш код здесь
text = reviews[0][0]

In [ ]:
#@title Решение (дважды щёлкните, чтобы раскрыть)
text = reviews[0][0]
print(text.lower())
print(len(text.split()))
print(text.split()[0])

## 1.2. Списки

Список (`list`) — упорядоченная изменяемая коллекция. Индексация с нуля, отрицательные индексы считаются с конца.

In [ ]:
nums = [5, 3, 8, 1]

print(nums[0], nums[-1], nums[1:3])
nums.append(10)        # добавить в конец
nums.insert(0, 100)    # вставить по индексу
nums.remove(8)         # удалить по значению
last = nums.pop()      # удалить и вернуть последний
print(nums, "удалили:", last)

print(len(nums), min(nums), max(nums), sum(nums))
print(3 in nums)

print(sorted(nums))    # новый отсортированный список, исходный не меняется
nums.sort()            # сортирует сам список «на месте»
print(nums)

In [ ]:
# ⚠️ Ловушка: присваивание не копирует список
a = [1, 2, 3]
b = a          # b — это ТОТ ЖЕ список
b.append(4)
print(a)       # [1, 2, 3, 4] — изменился и a!

c = a.copy()   # настоящая копия
c.append(5)
print(a, c)

# Кортеж (tuple) — неизменяемый список. Наши отзывы хранятся как кортежи (текст, метка)
text, label = reviews[0]   # распаковка
print(label, "—", text)

✍️ **Упражнение 1.2.** Напишите код, который из `reviews[15][0]` получает список слов:
- в нижнем регистре;
- без знаков препинания по краям слова (подсказка: `word.strip(".,!?—")`);
- только слова длиной от 3 букв.

Используйте цикл `for` и `append`.

In [ ]:
# ✍️ Ваш код здесь
text = reviews[15][0]
clean = []

In [ ]:
#@title Решение (дважды щёлкните, чтобы раскрыть)
text = reviews[15][0]
clean = []
for word in text.split():
    w = word.strip(".,!?—").lower()
    if len(w) >= 3:
        clean.append(w)
print(clean)

## 1.3. Словари и множества

Словарь (`dict`) хранит пары **ключ → значение**. Это главная структура для работы с API:
JSON, который возвращают все веб-сервисы, — это вложенные словари и списки.

In [ ]:
product = {"name": "Чайник", "price": 2490, "tags": ["кухня", "техника"]}

print(product["name"])
print(product.get("color", "не указан"))   # безопасный доступ со значением по умолчанию
product["color"] = "белый"                 # добавить или изменить ключ
print("price" in product)

for key, value in product.items():
    print(key, "→", value)

In [ ]:
import json

# Так выглядит ответ LLM API (упрощённо). Это строка с JSON...
raw = '{"choices": [{"message": {"role": "assistant", "content": "pos"}}], "usage": {"total_tokens": 42}}'

# ...которая превращается в обычные dict и list
data = json.loads(raw)
print(type(data))
print(data["choices"][0]["message"]["content"])   # путь к ответу модели
print(data["usage"]["total_tokens"])

In [ ]:
# Множество (set) — только уникальные элементы
words = "быстро очень быстро и очень удобно".split()
print(set(words))
print(len(words), "слов, из них уникальных:", len(set(words)))

## 1.4. Ветвления, циклы, функции

In [ ]:
score = 2
if score > 0:
    print("позитив")
elif score < 0:
    print("негатив")
else:
    print("нейтрально")

# Пустые коллекции, 0, "" и None считаются ложью
items = []
if not items:
    print("список пуст")

In [ ]:
# for по списку, range, enumerate
for i in range(3):
    print("итерация", i)

for i, (text, label) in enumerate(reviews[:3]):
    print(i, label, text)

# while и break
n = 1
while True:
    n *= 2
    if n > 100:
        break
print(n)

In [ ]:
# List comprehension — компактная запись цикла, который строит список
text = reviews[0][0]
long_words = [w.lower() for w in text.split() if len(w) > 5]
print(long_words)

# Функции
def clean_words(text):
    """Слова в нижнем регистре без знаков препинания."""
    return [w.strip(".,!?—«»").lower() for w in text.split() if w.strip(".,!?—«»")]

print(clean_words("Отличный чайник, закипает быстро!"))

✍️ **Упражнение 1.3. Частотный словарь (bag-of-words).** Посчитайте, сколько раз каждое слово встречается
во **всех** отзывах `reviews`, и выведите 5 самых частых слов длиной от 4 букв.

Подсказка: `counts[w] = counts.get(w, 0) + 1`, сортировка — `sorted(counts.items(), key=lambda kv: kv[1], reverse=True)`.

In [ ]:
# ✍️ Ваш код здесь
counts = {}

In [ ]:
#@title Решение (дважды щёлкните, чтобы раскрыть)
counts = {}
for text, _ in reviews:
    for w in clean_words(text):
        if len(w) >= 4:
            counts[w] = counts.get(w, 0) + 1

top = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)[:5]
for word, n in top:
    print(word, n)

✍️ **Упражнение 1.4. Наивный классификатор тональности (rule-based NLP).** Напишите функцию `naive_sentiment(text)`:
она считает слова из `POS` и `NEG` и возвращает `"pos"`, `"neg"` или `"neutral"`.

In [ ]:
POS = {"отличный", "хороший", "быстро", "удобный", "доволен", "рекомендую", "супер", "советую"}
NEG = {"сломался", "брак", "ужасный", "плохой", "долго", "кошмар", "протекает"}

In [ ]:
# ✍️ Ваш код здесь
def naive_sentiment(text):
    ...

In [ ]:
#@title Решение (дважды щёлкните, чтобы раскрыть)
def naive_sentiment(text):
    words = clean_words(text)
    score = sum(w in POS for w in words) - sum(w in NEG for w in words)
    if score > 0:
        return "pos"
    if score < 0:
        return "neg"
    return "neutral"

In [ ]:
# Проверим на всех отзывах
correct = 0
for text, label in reviews:
    pred = naive_sentiment(text)
    correct += pred == label
    if pred != label:
        print(f"✗ {pred:8} | {text}")
print(f"\nТочность: {correct}/{len(reviews)}")

**Посмотрите на ошибки.** «Отличное качество» не нашлось — в словаре только «отличный».
Пришлось бы перечислять все формы: *отличный, отличная, отличное, отличного…*

В русском языке у слова десятки форм. Эту проблему решает **морфологический анализ** — переходим ко второй части.

# Часть 2. Классический NLP

Так задачи обработки текста решали до появления LLM. Запускайте ячейки и меняйте входные тексты.

## 2.1. Морфология: pymorphy3

Лемматизация приводит слово к начальной форме: *отличная → отличный*, *сломался → сломаться*.

In [ ]:
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

p = morph.parse("отличная")[0]
print(p.normal_form, p.tag)    # начальная форма и грамматические признаки

def lemmas(text):
    return [morph.parse(w)[0].normal_form for w in clean_words(text)]

print(lemmas("Отличная доставка, чайник хороший"))
print(lemmas("Сломался через неделю, ужасное качество"))

In [ ]:
# Классификатор v2: те же правила, но по леммам
POS_L = {"отличный", "хороший", "быстро", "удобный", "довольный", "рекомендовать", "супер", "советовать"}
NEG_L = {"сломаться", "брак", "ужасный", "плохой", "долго", "кошмар", "протекать"}

def sentiment_v2(text):
    ls = lemmas(text)
    score = sum(l in POS_L for l in ls) - sum(l in NEG_L for l in ls)
    return "pos" if score > 0 else "neg" if score < 0 else "neutral"

for name, data in [("обучающие", reviews), ("тестовые", test_reviews)]:
    acc = sum(sentiment_v2(t) == l for t, l in data)
    print(f"{name}: {acc}/{len(data)}")

for t, l in test_reviews:
    print(f"{l} → {sentiment_v2(t):8} | {t}")

Стало лучше, но «Ни разу не пожалел о покупке» и «не советую» правила не понимают: нужно отдельно обрабатывать отрицания,
сарказм, противопоставления «но»… Каждый случай — новое правило.

## 2.2. Извлечение сущностей: Natasha + регулярные выражения

Задача из лекции: достать реквизиты из текста договора.

In [ ]:
print(contract_text)

In [ ]:
from natasha import Segmenter, NewsEmbedding, NewsNERTagger, Doc

segmenter = Segmenter()
ner_tagger = NewsNERTagger(NewsEmbedding())

doc = Doc(contract_text)
doc.segment(segmenter)
doc.tag_ner(ner_tagger)

for span in doc.spans:
    print(f"{span.type:4} {span.text}")    # PER — персона, ORG — организация, LOC — место

In [ ]:
import re

date = re.search(r"\d{2}\.\d{2}\.\d{4}", contract_text)
number = re.search(r"№\s*([\w/-]+)", contract_text)
amount = re.search(r"сумму\s+([\d\s]+)\s*руб", contract_text)

result = {
    "number": number.group(1) if number else None,
    "date": date.group() if date else None,
    "amount": int(amount.group(1).replace(" ", "")) if amount else None,
    "orgs": [s.text for s in doc.spans if s.type == "ORG"],
}
print(json.dumps(result, ensure_ascii=False, indent=2))

**Посмотрите внимательно на результат.** NER ошибается: у первой организации потерялась закрывающая кавычка,
слово «Поставщик» распознано как место (LOC), а «(Покупатель)» приклеилось к названию второй организации.
Модель Natasha обучена на новостях, а не на договорах.

**Вопрос:** организации найдены — но кто из них поставщик, а кто покупатель?
Для этого понадобятся новые правила: искать слово «(Поставщик)» после названия, учитывать «в лице», «именуемое в дальнейшем»…

✍️ Поменяйте текст договора: напишите сумму как «1,24 млн рублей» или дату как «3 февраля 2026 г.» — и посмотрите, что сломается.

## 2.3. Классификатор на машинном обучении: TF-IDF + логистическая регрессия

Вместо ручных правил модель **учится на размеченных примерах**.
TF-IDF превращает текст в вектор чисел (вес каждого слова), логистическая регрессия учится разделять классы.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

train_texts = [" ".join(lemmas(t)) for t, _ in reviews]   # учим на леммах
train_labels = [l for _, l in reviews]

clf = make_pipeline(TfidfVectorizer(), LogisticRegression())
clf.fit(train_texts, train_labels)

for t, l in test_reviews:
    pred = clf.predict([" ".join(lemmas(t))])[0]
    mark = "✓" if pred == l else "✗"
    print(f"{mark} {l} → {pred} | {t}")

In [ ]:
# Какие слова модель считает самыми позитивными и негативными
vec = clf.named_steps["tfidfvectorizer"]
lr = clf.named_steps["logisticregression"]
weights = sorted(zip(lr.coef_[0], vec.get_feature_names_out()))
print("negative:", [w for _, w in weights[:8]])
print("positive:", [w for _, w in weights[-8:]])

✍️ Придумайте 2–3 отзыва, на которых модель ошибается. Почему так происходит?

**Выводы части 2**
- Правила и классические модели **быстрые и дешёвые**: предсказание занимает доли миллисекунды.
- Но им нужны **ручные правила или размеченные данные**, а 30 примеров — очень мало.
- Сложные формулировки («но», отрицания, ирония) им даются плохо.

# Часть 3. Запрос к LLM через `requests`

LLM-приложение «под капотом» — это HTTP-запрос с JSON. Разберём HTTP на простом примере, затем обратимся к модели.

**HTTP-запрос** состоит из: метода (`GET` — получить, `POST` — отправить), URL, заголовков (например, ключ доступа) и тела (данные в JSON).
Ответ содержит **статус-код** (`200` — успех, `401` — нет доступа, `429` — превышен лимит, `5xx` — ошибка сервера) и тело.

## 3.1. Разминка: GET-запрос к публичному API

In [ ]:
import requests

r = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": 57.77, "longitude": 40.93, "current": "temperature_2m,wind_speed_10m"},
    timeout=10,
)
print("Статус:", r.status_code)
weather = r.json()           # JSON → dict
print(weather["current"])
print(f"В Костроме сейчас {weather['current']['temperature_2m']} °C")

## 3.2. Настройка доступа к LLM

Используем **OpenAI-совместимый** формат запроса (`/v1/chat/completions`). Его поддерживают большинство провайдеров
(OpenRouter, облачные платформы, локальная Ollama), поэтому код переносится между ними заменой URL и названия модели.

**Ключ API никогда не пишем в код.** В Colab: значок 🔑 **Secrets** на левой панели → добавить секрет `LLM_API_KEY` → включить доступ для ноутбука.

In [ ]:
# ⚙️ Настройки — заполните под своего провайдера (их выдаст преподаватель)
API_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "<название модели>"
# Пример для локальной Ollama:
# API_URL = "http://<адрес-сервера>:11434/v1/chat/completions"
# MODEL = "qwen2.5:7b"

try:
    from google.colab import userdata
    API_KEY = userdata.get("LLM_API_KEY")
except Exception:
    from getpass import getpass
    API_KEY = getpass("Введите ключ API: ")

print("Ключ загружен:", bool(API_KEY))

## 3.3. Первый запрос

In [ ]:
payload = {
    "model": MODEL,
    "temperature": 0,     # минимум случайности в ответе
    "messages": [
        {"role": "system", "content": "Определи тональность отзыва покупателя. Ответь одним словом: pos или neg."},
        {"role": "user", "content": "Привезли быстро, но через неделю он сломался"},
    ],
}
headers = {"Authorization": f"Bearer {API_KEY}"}

r = requests.post(API_URL, headers=headers, json=payload, timeout=60)
print("Статус:", r.status_code)
r.raise_for_status()     # если статус не 2xx — выбросить ошибку

data = r.json()
print("Ответ модели:", data["choices"][0]["message"]["content"])
print("Токены:", data.get("usage"))

Посмотрите на структуру ответа целиком — это те самые вложенные словари и списки из части 1:

In [ ]:
print(json.dumps(data, ensure_ascii=False, indent=2))

## 3.4. Оборачиваем в функцию и сравниваем с классическим подходом

In [ ]:
import time

def ask_llm(system, user, temperature=0):
    payload = {
        "model": MODEL,
        "temperature": temperature,
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
    }
    r = requests.post(API_URL, headers={"Authorization": f"Bearer {API_KEY}"}, json=payload, timeout=60)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()

SENTIMENT_PROMPT = "Определи тональность отзыва покупателя. Ответь одним словом: pos или neg."

print(f"{'метка':5} {'правила':8} {'TF-IDF':7} {'LLM':5} отзыв")
for t, l in test_reviews:
    start = time.perf_counter()
    llm_pred = ask_llm(SENTIMENT_PROMPT, t).lower()
    ms = (time.perf_counter() - start) * 1000
    tfidf_pred = clf.predict([" ".join(lemmas(t))])[0]
    print(f"{l:5} {sentiment_v2(t):8} {tfidf_pred:7} {llm_pred:5} {t}  [{ms:.0f} мс]")

**Обратите внимание на время:** LLM отвечает за сотни миллисекунд или секунды. TF-IDF-модель — за доли миллисекунды:

In [ ]:
import time
start = time.perf_counter()
for _ in range(1000):
    clf.predict([" ".join(lemmas(test_reviews[0][0]))])
print(f"TF-IDF: {(time.perf_counter() - start):.3f} мс на отзыв (среднее по 1000)")

## 3.5. Извлечение данных из договора

Попросим модель вернуть JSON и разберём его через `json.loads`.

In [ ]:
EXTRACT_PROMPT = """Извлеки реквизиты из текста договора.
Верни только JSON с полями:
number (строка), date (YYYY-MM-DD), supplier (строка), buyer (строка), amount (число, рубли).
Если поля нет в тексте — null."""

answer = ask_llm(EXTRACT_PROMPT, contract_text)
print("Сырой ответ:\n", answer)

contract = json.loads(answer)
print("\nПоставщик:", contract["supplier"])
print("Покупатель:", contract["buyer"])

Если ячейка выше упала с `JSONDecodeError` — модель, скорее всего, обернула JSON в блок markdown:
` ```json ... ``` `. Это типичная проблема, вот простое решение:

In [ ]:
def parse_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        text = text[text.find("{"):]          # отрезаем пометку json
    return json.loads(text)

contract = parse_json(answer)
print(json.dumps(contract, ensure_ascii=False, indent=2))

Сравните с частью 2.2: роли сторон (поставщик / покупатель) модель поняла **из контекста**, без единого правила.
Но гарантий нет — ответ нужно проверять. Надёжные способы (structured output, валидация схемой) разберём на следующих занятиях.

✍️ Измените текст договора: сумма прописью, дата словами, стороны в обратном порядке. Справляется ли модель?

# Итоги

| Подход | Плюсы | Минусы |
|---|---|---|
| Правила | Быстро, прозрачно | Хрупко, каждый случай — новое правило |
| Классический ML | Быстро и дёшево | Нужны размеченные данные |
| LLM | Без разметки, понимает нюансы и контекст | Медленно, платно, ответ нужно проверять |

## Домашнее задание
1. Установите Python локально (VS Code + `uv` или `venv`), сохраните функцию `clean_words` в файл `.py` и запустите его из терминала.
2. Напишите функцию: на вход текст, на выход частотный словарь **лемм** без стоп-слов (*и, в, на, не, что…*).
3. Через `ask_llm` классифицируйте все 30 отзывов из `reviews` и сравните с TF-IDF-моделью: на каких отзывах они расходятся и почему?